# 📚 Smart Library — Couche Silver
## Notebook 02 : Nettoyage, Enrichissement et UPSERT Iceberg (MERGE INTO)

---

### Objectif
Transformer les données brutes de la couche Bronze en données **propres, enrichies et fiables**.

### Transformations effectuées
| Table Bronze | Table Silver | Opérations |
|---|---|---|
| `bronze.utilisateurs` | `silver.utilisateurs` | Déduplication, normalisation email/ville |
| `bronze.livres` | `silver.livres` | Enrichissement catégorie, score qualité |
| `bronze.commandes` | `silver.commandes` | Jointure livres+users, calculs montants |
| `bronze.stock_cow` | `silver.stock` | UPSERT (MERGE INTO) temps réel, alertes stock |
| `bronze.catalogue` | `silver.catalogue` | Nettoyage, déduplications |

### Concepts Iceberg démontrés
- **UPSERT via `MERGE INTO`** : mise à jour ciblée sans réécriture complète de la table
  (contrairement à Hudi, il n'y a pas d'upsert *implicite* — il est déclaré explicitement en SQL)
- **Schema evolution** : ajout de colonnes sans casser la table (`ALTER TABLE ... ADD COLUMN`)
- **Changelog / snapshots** : lire uniquement ce qui a changé depuis un snapshot donné,
  via la procédure système `create_changelog_view`

---
## Étape 0 — Initialisation SparkSession

In [1]:
import os, sys, time, re
from datetime import datetime
from pathlib import Path

# ── Tuer toute session existante ─────────────────────────────
try:
    from pyspark.sql import SparkSession
    SparkSession.getActiveSession().stop()
except:
    pass

# ── Java ──────────────────────────────────────────────────────
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"]      = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

# ── Jars Iceberg + MinIO ──────────────────────────────────────
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.5.2,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Connexion MinIO + Lakekeeper ──────────────────────────────
MINIO_ENDPOINT       = "http://minio:9000"
MINIO_ACCESS         = "minioadmin"
MINIO_SECRET         = "minioadmin123"
CATALOG_NAME         = "lakehouse"
LAKEKEEPER_URI       = "http://lakekeeper:8181/catalog"
LAKEKEEPER_WAREHOUSE = "lakehouse"

NS_BRONZE = f"{CATALOG_NAME}.bronze"
NS_SILVER = f"{CATALOG_NAME}.silver"
NS_GOLD   = f"{CATALOG_NAME}.gold"

# ── Fonctions utilitaires Iceberg (équivalent hudi_options) ───
#
#  Avec Hudi : le type de table (COW/MOR) et la clé d'upsert se
#  déclarent au niveau de l'ÉCRITURE (options du DataFrameWriter).
#  Avec Iceberg : ce sont des PROPRIÉTÉS DE TABLE, et l'UPSERT
#  s'exprime explicitement en SQL via MERGE INTO.
#
def iceberg_table_properties(type_table="copy-on-write"):
    """
    type_table :
      - "copy-on-write" → équivalent Hudi COW
      - "merge-on-read"  → équivalent Hudi MOR
    """
    return {
        "format-version"    : "2",
        "write.delete.mode" : type_table,
        "write.update.mode" : type_table,
        "write.merge.mode"  : type_table,
    }

def create_or_replace_table(df, table_name, partition_cols=None, type_table="copy-on-write"):
    """Équivalent de : df.write.format('hudi')....mode('overwrite').save(chemin)"""
    # Iceberg (ClusteredWriter) exige que les lignes soient triées/regroupées
    # par colonne de partition AVANT l'écriture — contrairement à Hudi qui
    # gère le clustering en interne. Sans ce tri : "Incoming records violate
    # the writer assumption that records are clustered by spec and partition".
    if partition_cols:
        df = df.sort(*partition_cols)
    writer = df.writeTo(table_name)
    if partition_cols:
        writer = writer.partitionedBy(*[F.col(c) for c in partition_cols])
    for k, v in iceberg_table_properties(type_table).items():
        writer = writer.tableProperty(k, v)
    writer.createOrReplace()

def append_table(df, table_name):
    """Ajout de lignes (insert simple, pas d'upsert)."""
    df.writeTo(table_name).append()

def upsert_table(spark, df, table_name, key_cols):
    """UPSERT via MERGE INTO — équivalent fonctionnel de l'UPSERT automatique Hudi."""
    vue = f"_staging_{table_name.split('.')[-1]}"
    df.createOrReplaceTempView(vue)
    condition = " AND ".join([f"t.{c} = s.{c}" for c in key_cols])
    spark.sql(f"""
        MERGE INTO {table_name} t
        USING {vue} s
        ON {condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def snapshots(spark, table_name):
    """Retourne l'historique des snapshots Iceberg (équivalent de la timeline Hudi)."""
    return spark.sql(f"SELECT * FROM {table_name}.snapshots ORDER BY committed_at")

# ── SparkSession ──────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("SmartLibrary-Silver")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", LAKEKEEPER_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LAKEKEEPER_WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint",           MINIO_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id",     MINIO_ACCESS)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", MINIO_SECRET)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config("spark.serializer",
            "org.apache.spark.serializer.KryoSerializer")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ── Création des namespaces Medallion (idempotent) ────────────
for ns in (NS_BRONZE, NS_SILVER, NS_GOLD):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ns}")

print(f"✅ SparkSession démarrée — version {spark.version}")
print(f"   Catalogue Iceberg : {CATALOG_NAME} (REST → Lakekeeper → Postgres)")
print(f"   Stockage          : {MINIO_ENDPOINT} (bucket lakehouse)")

# ── Tables Bronze (lecture) ────────────────────────────────────
BRONZE_UTILISATEURS = f"{NS_BRONZE}.utilisateurs"
BRONZE_LIVRES       = f"{NS_BRONZE}.livres"
BRONZE_COMMANDES    = f"{NS_BRONZE}.commandes"
BRONZE_STOCK        = f"{NS_BRONZE}.stock_cow"
BRONZE_CATALOGUE    = f"{NS_BRONZE}.catalogue"

# ── Tables Silver (écriture) ───────────────────────────────────
SILVER_UTILISATEURS = f"{NS_SILVER}.utilisateurs"
SILVER_LIVRES       = f"{NS_SILVER}.livres"
SILVER_COMMANDES    = f"{NS_SILVER}.commandes"
SILVER_STOCK        = f"{NS_SILVER}.stock"
SILVER_CATALOGUE    = f"{NS_SILVER}.catalogue"

print(f"   Source      : {NS_BRONZE}")
print(f"   Destination : {NS_SILVER}")

✅ SparkSession démarrée — version 3.4.0
   Catalogue Iceberg : lakehouse (REST → Lakekeeper → Postgres)
   Stockage          : http://minio:9000 (bucket lakehouse)
   Source      : lakehouse.bronze
   Destination : lakehouse.silver


---
## Étape 1 — Silver Utilisateurs
### Opérations : déduplication, normalisation, statut enrichi

In [2]:
# ── Lecture Bronze ────────────────────────────────────────────
# (contrairement à Hudi, une table Iceberg n'expose pas de colonnes
#  techniques _hoodie_* : les métadonnées vivent dans le catalogue,
#  pas dans les lignes de données — rien à "drop" ici)
df_users_bronze = spark.table(BRONZE_UTILISATEURS)

print(f"📋 Bronze utilisateurs : {df_users_bronze.count()} lignes")

# ── Vérification doublons ─────────────────────────────────────
nb_doublons = df_users_bronze.count() - df_users_bronze.dropDuplicates(["utilisateur_id"]).count()
print(f"   Doublons détectés sur utilisateur_id : {nb_doublons}")

📋 Bronze utilisateurs : 1000 lignes
   Doublons détectés sur utilisateur_id : 0


In [3]:
# ── Nettoyage et enrichissement ───────────────────────────────
df_users_silver = (
    df_users_bronze
    # Déduplication — garder la ligne la plus récente
    .dropDuplicates(["utilisateur_id"])
    # Normalisation email en minuscules
    .withColumn("email",         F.lower(F.trim(F.col("email"))))
    # Normalisation ville
    .withColumn("ville",         F.initcap(F.trim(F.col("ville"))))
    # Normalisation pays
    .withColumn("pays",          F.initcap(F.trim(F.col("pays"))))
    # Supprimer lignes sans email valide
    .filter(F.col("email").rlike(r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"))
    # Remplacer nulls
    .fillna({"genre_prefere": "Non renseigné", "ville": "Inconnue"})
    # Colonne segment utilisateur
    .withColumn("segment", F.when(F.col("nb_commandes_total") >= 20, F.lit("VIP"))
                            .when(F.col("nb_commandes_total") >= 5,  F.lit("Régulier"))
                            .otherwise(F.lit("Nouveau")))
    # Ancienneté en jours
    .withColumn("anciennete_jours",
                F.datediff(F.current_date(), F.to_date(F.col("date_inscription"))))
    # Traçabilité Silver
    .withColumn("_couche",         F.lit("silver"))
    .withColumn("updated_at",      F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_pays",  F.col("pays"))
)

print(f"✅ Silver utilisateurs : {df_users_silver.count()} lignes après nettoyage")
print()
print("Distribution par segment :")
df_users_silver.groupBy("segment").count().orderBy("count", ascending=False).show()

df_users_silver.select(
    "utilisateur_id", "nom", "email", "pays", "segment", "anciennete_jours"
).show(5, truncate=False)

✅ Silver utilisateurs : 1000 lignes après nettoyage

Distribution par segment :
+--------+-----+
| segment|count|
+--------+-----+
|     VIP|  614|
|Régulier|  283|
| Nouveau|  103|
+--------+-----+

+--------------+-------+----------------------------+-------+-------+----------------+
|utilisateur_id|nom    |email                       |pays   |segment|anciennete_jours|
+--------------+-------+----------------------------+-------+-------+----------------+
|USR00001      |Peltier|denisguerin@example.com     |Maroc  |VIP    |651             |
|USR00002      |Mallet |margotbouvier@example.org   |Tunisie|VIP    |775             |
|USR00003      |Gimenez|duhamelmichele@example.com  |Maroc  |VIP    |580             |
|USR00004      |Bertin |martin66@example.com        |Tunisie|VIP    |658             |
|USR00005      |Lacroix|isabellepasquier@example.org|Maroc  |VIP    |572             |
+--------------+-------+----------------------------+-------+-------+----------------+
only showing top 

In [4]:
# ── Écriture Iceberg Silver ────────────────────────────────────
create_or_replace_table(
    df_users_silver, SILVER_UTILISATEURS,
    partition_cols=["partition_pays"], type_table="copy-on-write"
)

n = spark.table(SILVER_UTILISATEURS).count()
print(f"✅ {SILVER_UTILISATEURS} — {n} lignes écrites dans Iceberg")

✅ lakehouse.silver.utilisateurs — 1000 lignes écrites dans Iceberg


---
## Étape 2 — Silver Livres
### Opérations : enrichissement, score qualité, catégorisation prix

In [5]:
# ── Lecture Bronze ────────────────────────────────────────────
df_livres_bronze = spark.table(BRONZE_LIVRES)

print(f"📚 Bronze livres : {df_livres_bronze.count()} lignes")

# ── Nettoyage et enrichissement ───────────────────────────────
df_livres_silver = (
    df_livres_bronze
    .dropDuplicates(["livre_id"])
    # Normalisation
    .withColumn("titre",   F.initcap(F.trim(F.col("titre"))))
    .withColumn("auteur",  F.initcap(F.trim(F.col("auteur"))))
    .withColumn("editeur", F.initcap(F.trim(F.col("editeur"))))
    # Supprimer prix aberrants
    .filter(F.col("prix_vente") > 0)
    .filter(F.col("nb_pages")   > 0)
    # Catégorie de prix
    .withColumn("categorie_prix",
                F.when(F.col("prix_vente") <= 10, F.lit("Économique"))
                 .when(F.col("prix_vente") <= 25, F.lit("Standard"))
                 .when(F.col("prix_vente") <= 40, F.lit("Premium"))
                 .otherwise(F.lit("Luxe")))
    # Score popularité = note * log(nb_avis + 1)
    .withColumn("score_popularite",
                F.round(
                    F.col("note_moyenne") * F.log(F.col("nb_avis").cast("double") + 1), 2
                ))
    # Marge bénéficiaire
    .withColumn("marge_pct",
                F.round(
                    (F.col("prix_vente") - F.col("prix_achat")) / F.col("prix_vente") * 100, 1
                ))
    # Disponibilité
    .withColumn("est_disponible", F.col("disponible").cast("boolean"))
    # Traçabilité
    .withColumn("_couche",         F.lit("silver"))
    .withColumn("updated_at",      F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_genre", F.col("genre"))
)

print(f"✅ Silver livres : {df_livres_silver.count()} lignes")
print()
print("Répartition par catégorie de prix :")
df_livres_silver.groupBy("categorie_prix").count().orderBy("categorie_prix").show()

df_livres_silver.select(
    "livre_id", "titre", "genre", "prix_vente",
    "categorie_prix", "score_popularite", "marge_pct"
).orderBy("score_popularite", ascending=False).show(5, truncate=True)

📚 Bronze livres : 500 lignes
✅ Silver livres : 500 lignes

Répartition par catégorie de prix :
+--------------+-----+
|categorie_prix|count|
+--------------+-----+
|          Luxe|  116|
|       Premium|  199|
|      Standard|  185|
+--------------+-----+

+--------+--------------------+--------------------+----------+--------------+----------------+---------+
|livre_id|               titre|               genre|prix_vente|categorie_prix|score_popularite|marge_pct|
+--------+--------------------+--------------------+----------+--------------+----------------+---------+
|  BK0306|        Secret Parti|Développement per...|     32.51|       Premium|           30.55|     55.5|
|  BK0061|Etc Contraire Lon...|              Poésie|     34.54|       Premium|           30.04|     60.2|
|  BK0329|Banc Alors Suppos...|         Philosophie|     41.57|          Luxe|           29.98|     53.0|
|  BK0371|Essuyer Attacher ...|              Poésie|     32.67|       Premium|            29.9|     66.2|
|

In [6]:
# ── Écriture Iceberg Silver ────────────────────────────────────
create_or_replace_table(
    df_livres_silver, SILVER_LIVRES,
    partition_cols=["partition_genre"], type_table="copy-on-write"
)

n = spark.table(SILVER_LIVRES).count()
print(f"✅ {SILVER_LIVRES} — {n} lignes écrites dans Iceberg")

✅ lakehouse.silver.livres — 500 lignes écrites dans Iceberg


---
## Étape 3 — Silver Commandes
### Opérations : jointure livres + utilisateurs, calculs enrichis

> 💡 C'est ici qu'on **enrichit les commandes** avec les infos des livres et des utilisateurs
> pour avoir une table prête pour l'analyse dans la couche Gold.

In [7]:
# ── Lecture des trois tables ───────────────────────────────────
df_cmd = spark.table(BRONZE_COMMANDES)
df_liv = (
    spark.table(SILVER_LIVRES)
    .select("livre_id", "titre", "genre", "auteur",
            "editeur", "prix_vente", "categorie_prix")
)
df_usr = (
    spark.table(SILVER_UTILISATEURS)
    .select("utilisateur_id", "nom", "ville", "pays", "segment")
)

print(f"🛒 Commandes Bronze : {df_cmd.count()} lignes")
print(f"📚 Livres Silver    : {df_liv.count()} lignes")
print(f"👥 Utilisateurs     : {df_usr.count()} lignes")

🛒 Commandes Bronze : 5000 lignes
📚 Livres Silver    : 500 lignes
👥 Utilisateurs     : 1000 lignes


In [8]:
# ── Jointures + enrichissement ────────────────────────────────
df_commandes_silver = (
    df_cmd
    # Jointure avec livres
    .join(df_liv, on="livre_id", how="left")
    # Jointure avec utilisateurs
    .join(df_usr, on="utilisateur_id", how="left")
    # Nettoyage montants
    .filter(F.col("montant_total") > 0)
    # Calcul TVA (19%)
    .withColumn("montant_ht",
                F.round(F.col("montant_total") / 1.19, 2))
    .withColumn("montant_tva",
                F.round(F.col("montant_total") - F.col("montant_ht"), 2))
    # Délai de livraison en jours
    .withColumn("delai_livraison_jours",
                F.when(
                    F.col("date_livraison").isNotNull() & (F.col("date_livraison") != ""),
                    F.datediff(
                        F.to_date(F.col("date_livraison")),
                        F.to_date(F.col("date_commande"))
                    )
                ).otherwise(F.lit(None)))
    # Tranche horaire commande
    .withColumn("heure_commande",
                F.hour(F.to_timestamp(F.col("date_commande"))))
    .withColumn("tranche_horaire",
                F.when(F.col("heure_commande").between(6, 11),  F.lit("Matin"))
                 .when(F.col("heure_commande").between(12, 17), F.lit("Après-midi"))
                 .when(F.col("heure_commande").between(18, 22), F.lit("Soirée"))
                 .otherwise(F.lit("Nuit")))
    # Traçabilité
    .withColumn("_couche",           F.lit("silver"))
    .withColumn("updated_at",        F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_periode", F.col("partition_periode"))
)

print(f"✅ Silver commandes enrichies : {df_commandes_silver.count()} lignes")
print()
print("Répartition par tranche horaire :")
df_commandes_silver.groupBy("tranche_horaire").count().orderBy("count", ascending=False).show()

df_commandes_silver.select(
    "commande_id", "titre", "nom", "pays",
    "montant_total", "montant_ht", "montant_tva",
    "statut", "tranche_horaire"
).show(5, truncate=True)

✅ Silver commandes enrichies : 5000 lignes

Répartition par tranche horaire :
+---------------+-----+
|tranche_horaire|count|
+---------------+-----+
|           Nuit| 1505|
|          Matin| 1252|
|     Après-midi| 1221|
|         Soirée| 1022|
+---------------+-----+

+-----------+--------------------+-------+-------+-------------+----------+-----------+--------+---------------+
|commande_id|               titre|    nom|   pays|montant_total|montant_ht|montant_tva|  statut|tranche_horaire|
+-----------+--------------------+-------+-------+-------------+----------+-----------+--------+---------------+
| ORD0000019|Paquet Fidèle Ins...| Samson|Tunisie|        27.66|     23.24|       4.42|en_cours|         Soirée|
| ORD0000222|Passer Avant Teni...|Le Gall|Tunisie|         39.5|     33.19|       6.31|   livré|          Matin|
| ORD0000229|Adresser Intentio...| Bailly|  Maroc|        47.28|     39.73|       7.55|  annulé|           Nuit|
| ORD0000243|Désigner Menacer ...|   Lamy|  Maroc| 

In [9]:
# ── Écriture Iceberg Silver ────────────────────────────────────
create_or_replace_table(
    df_commandes_silver, SILVER_COMMANDES,
    partition_cols=["partition_periode"], type_table="copy-on-write"
)

n = spark.table(SILVER_COMMANDES).count()
print(f"✅ {SILVER_COMMANDES} — {n} lignes écrites dans Iceberg")

✅ lakehouse.silver.commandes — 5000 lignes écrites dans Iceberg


---
## Étape 4 — Silver Stock avec UPSERT Iceberg (`MERGE INTO`)

> 💡 **Démonstration clé** : le stock change en permanence (ventes, livraisons).
> Avec `MERGE INTO`, on met à jour **uniquement les lignes modifiées** sans réécrire toute la table.
> C'est impossible avec du Parquet brut — c'est ce qui fait la valeur du Lakehouse.
>
> Différence avec Hudi : Hudi déclenche l'upsert *automatiquement* dès qu'on écrit en mode
> `append` (grâce à `hoodie.datasource.write.recordkey.field`). Avec Iceberg, l'upsert est
> **un statement SQL explicite** — plus verbeux, mais aussi plus lisible et plus proche du SQL standard.

In [10]:
# ── Lecture Bronze Stock ──────────────────────────────────────
df_stock_bronze = spark.table(BRONZE_STOCK)

# Jointure avec livres pour enrichir
df_livres_ref = (
    spark.table(SILVER_LIVRES)
    .select("livre_id", "titre", "genre", "prix_vente")
)

df_stock_silver = (
    df_stock_bronze
    .join(df_livres_ref, on="livre_id", how="left")
    .withColumn("statut_stock",
                F.when(F.col("quantite_disponible") == 0,   F.lit("Rupture"))
                 .when(F.col("quantite_disponible") <= F.col("seuil_alerte"), F.lit("Alerte"))
                 .when(F.col("quantite_disponible") <= 50,  F.lit("Faible"))
                 .otherwise(F.lit("Suffisant")))
    .withColumn("valeur_stock",
                F.round(F.col("quantite_disponible") * F.col("prix_vente"), 2))
    .withColumn("_couche",           F.lit("silver"))
    .withColumn("updated_at",        F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_entrepot",F.col("entrepot"))
)

print(f"📦 Stock Silver initial : {df_stock_silver.count()} lignes")
print()
print("Répartition par statut stock :")
df_stock_silver.groupBy("statut_stock").count().orderBy("count", ascending=False).show()

df_stock_silver.select(
    "stock_id", "titre", "quantite_disponible",
    "statut_stock", "valeur_stock", "entrepot"
).show(5, truncate=True)

📦 Stock Silver initial : 500 lignes

Répartition par statut stock :
+------------+-----+
|statut_stock|count|
+------------+-----+
|   Suffisant|  395|
|      Faible|   83|
|      Alerte|   21|
|     Rupture|    1|
+------------+-----+

+----------+--------------------+-------------------+------------+------------+--------+
|  stock_id|               titre|quantite_disponible|statut_stock|valeur_stock|entrepot|
+----------+--------------------+-------------------+------------+------------+--------+
|STK-BK0002|Mensonge Explique...|                 75|   Suffisant|      1060.5|    Sfax|
|STK-BK0006|       Ouvert Ancien|                117|   Suffisant|     3056.04|    Sfax|
|STK-BK0007|                 Île|                 14|      Alerte|      365.82|    Sfax|
|STK-BK0008|    Aider Oui Manger|                 78|   Suffisant|     3413.28|    Sfax|
|STK-BK0014|   Monde Nommer Cinq|                118|   Suffisant|     5810.32|    Sfax|
+----------+--------------------+------------------

In [11]:
# ── Écriture initiale Silver Stock ───────────────────────────
create_or_replace_table(
    df_stock_silver, SILVER_STOCK,
    partition_cols=["partition_entrepot"], type_table="copy-on-write"
)

n = spark.table(SILVER_STOCK).count()
print(f"✅ silver.stock initial — {n} lignes")

✅ silver.stock initial — 500 lignes


In [12]:
# ════════════════════════════════════════════════════════════
# DÉMONSTRATION UPSERT ICEBERG (MERGE INTO) — Simulation de ventes
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("  DÉMONSTRATION UPSERT ICEBERG — MERGE INTO")
print("  Simulation : 30 livres vendus → stock mis à jour")
print("=" * 60)

# Snapshot AVANT
snapshot_avant = spark.sql(f"SELECT snapshot_id FROM {SILVER_STOCK}.snapshots ORDER BY committed_at DESC LIMIT 1").first()["snapshot_id"]
print(f"\n  État AVANT — snapshot_id : {snapshot_avant}")
spark.table(SILVER_STOCK).select("stock_id", "titre", "quantite_disponible", "statut_stock") \
     .orderBy("quantite_disponible").show(5, truncate=True)

# Simuler 30 ventes : décrémenter le stock
df_ventes = (
    spark.table(SILVER_STOCK)
    .limit(30)
    .withColumn("quantite_disponible",
                F.greatest(F.col("quantite_disponible") - F.lit(5), F.lit(0)))
    .withColumn("statut_stock",
                F.when(F.col("quantite_disponible") == 0,   F.lit("Rupture"))
                 .when(F.col("quantite_disponible") <= F.col("seuil_alerte"), F.lit("Alerte"))
                 .when(F.col("quantite_disponible") <= 50,  F.lit("Faible"))
                 .otherwise(F.lit("Suffisant")))
    .withColumn("updated_at", F.lit(datetime.now().isoformat()).cast(StringType()))
)

# UPSERT — MERGE INTO ne touche que les 30 lignes concernées
debut = time.time()
upsert_table(spark, df_ventes, SILVER_STOCK, key_cols=["stock_id"])
temps_upsert = round(time.time() - debut, 2)

# Snapshot APRÈS
snapshot_apres = spark.sql(f"SELECT snapshot_id FROM {SILVER_STOCK}.snapshots ORDER BY committed_at DESC LIMIT 1").first()["snapshot_id"]

print(f"\n  État APRÈS  — snapshot_id : {snapshot_apres}")
spark.table(SILVER_STOCK).select("stock_id", "titre", "quantite_disponible", "statut_stock") \
     .orderBy("quantite_disponible").show(5, truncate=True)

print(f"  ✅ UPSERT (MERGE INTO) de 30 lignes effectué en {temps_upsert}s")
print(f"  Total lignes inchangées : {n - 30}")
print(f"  💡 Iceberg n'a réécrit QUE les fichiers contenant les 30 lignes concernées")
print(f"     (grâce au copy-on-write ciblé au niveau fichier, pas ligne)")

  DÉMONSTRATION UPSERT ICEBERG — MERGE INTO
  Simulation : 30 livres vendus → stock mis à jour

  État AVANT — snapshot_id : 1565405350769361658
+----------+--------------------+-------------------+------------+
|  stock_id|               titre|quantite_disponible|statut_stock|
+----------+--------------------+-------------------+------------+
|STK-BK0060|     Repas Confiance|                  0|     Rupture|
|STK-BK0445|        Quatre Barbe|                  1|      Alerte|
|STK-BK0411|            Enfoncer|                  1|      Alerte|
|STK-BK0127|            Inventer|                  1|      Alerte|
|STK-BK0113|Quoi Relation Cer...|                  2|      Alerte|
+----------+--------------------+-------------------+------------+
only showing top 5 rows


  État APRÈS  — snapshot_id : 1537247837498438725
+----------+--------------------+-------------------+------------+
|  stock_id|               titre|quantite_disponible|statut_stock|
+----------+--------------------+---------

---
## Étape 5 — Requête de changements (Changelog Iceberg)

> 💡 **Concept avancé** : au lieu de relire TOUTE la table à chaque pipeline,
> Iceberg permet de lire **uniquement ce qui a changé** entre deux snapshots,
> via la procédure système `create_changelog_view` — l'équivalent de la requête
> incrémentale (`_hoodie_commit_time > X`) de Hudi, mais basée sur les snapshots.

In [13]:
# ── Vue de changements via la procédure système Iceberg ───────
print("=" * 60)
print("  CHANGELOG ICEBERG")
print(f"  Changements entre le snapshot {snapshot_avant}")
print(f"  et le snapshot {snapshot_apres}")
print("=" * 60)

spark.sql(f"""
    CALL {CATALOG_NAME}.system.create_changelog_view(
        table   => 'silver.stock',
        options => map('start-snapshot-id', '{snapshot_avant}',
                        'end-snapshot-id',   '{snapshot_apres}')
    )
""")

df_incremental = spark.sql("""
    SELECT stock_id, titre, quantite_disponible, statut_stock,
           _change_type, _change_ordinal
    FROM   stock_changes
    ORDER BY _change_ordinal
""")

n_changes = df_incremental.count()
print(f"\n  Lignes modifiées depuis le snapshot précédent : {n_changes}")
print(f"  (sur {n} lignes totales dans la table)")
print()
df_incremental.show(10, truncate=True)

print("  💡 En production : ce mécanisme permet de ne traiter")
print("     que les données nouvelles → pipeline 10x plus rapide")

  CHANGELOG ICEBERG
  Changements entre le snapshot 1565405350769361658
  et le snapshot 1537247837498438725

  Lignes modifiées depuis le snapshot précédent : 60
  (sur 500 lignes totales dans la table)

+----------+--------------------+-------------------+------------+------------+---------------+
|  stock_id|               titre|quantite_disponible|statut_stock|_change_type|_change_ordinal|
+----------+--------------------+-------------------+------------+------------+---------------+
|STK-BK0002|Mensonge Explique...|                 70|   Suffisant|      INSERT|              0|
|STK-BK0002|Mensonge Explique...|                 75|   Suffisant|      DELETE|              0|
|STK-BK0006|       Ouvert Ancien|                112|   Suffisant|      INSERT|              0|
|STK-BK0006|       Ouvert Ancien|                117|   Suffisant|      DELETE|              0|
|STK-BK0007|                 Île|                  9|      Alerte|      INSERT|              0|
|STK-BK0007|               

---
## Étape 6 — Silver Catalogue

In [14]:
# ── Lecture et nettoyage ──────────────────────────────────────
df_cat_bronze = spark.table(BRONZE_CATALOGUE)

df_cat_silver = (
    df_cat_bronze
    .dropDuplicates(["catalogue_id"])
    .filter(F.col("livre_id").isNotNull())
    .withColumn("nom_categorie",      F.initcap(F.trim(F.col("nom_categorie"))))
    .withColumn("nom_sous_categorie", F.initcap(F.trim(F.col("nom_sous_categorie"))))
    .withColumn("_couche",            F.lit("silver"))
    .withColumn("updated_at",         F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_cat",      F.col("nom_categorie"))
)

create_or_replace_table(
    df_cat_silver, SILVER_CATALOGUE,
    partition_cols=["partition_cat"], type_table="copy-on-write"
)

n = spark.table(SILVER_CATALOGUE).count()
print(f"✅ {SILVER_CATALOGUE} — {n} lignes")

✅ lakehouse.silver.catalogue — 291 lignes


---
## Étape 7 — Bilan de la couche Silver

In [15]:
tables_silver = [
    ("silver.utilisateurs", SILVER_UTILISATEURS),
    ("silver.livres",       SILVER_LIVRES),
    ("silver.commandes",    SILVER_COMMANDES),
    ("silver.stock",        SILVER_STOCK),
    ("silver.catalogue",    SILVER_CATALOGUE),
]

print("=" * 55)
print(f"  {'TABLE SILVER':<28} {'LIGNES':>8}")
print("=" * 55)
total = 0
for nom, table in tables_silver:
    try:
        n = spark.table(table).count()
        total += n
        print(f"  {nom:<28} {n:>8,}  ✅")
    except Exception as e:
        print(f"  {nom:<28} {'ERREUR':>8}  ❌")
print("=" * 55)
print(f"  {'TOTAL':<28} {total:>8,}")
print("=" * 55)
print()
print("  Concepts Iceberg démontrés dans ce notebook :")
print("  ✅ UPSERT ciblé via MERGE INTO (30 lignes sur 500 modifiées)")
print("  ✅ Changelog inter-snapshots (lire uniquement les changements)")
print("  ✅ Enrichissement par jointures multi-tables")
print()
print("  ➡  Prochain : 03_analytique_gold.ipynb")

  TABLE SILVER                   LIGNES
  silver.utilisateurs             1,000  ✅
  silver.livres                     500  ✅
  silver.commandes                5,000  ✅
  silver.stock                      500  ✅
  silver.catalogue                  291  ✅
  TOTAL                           7,291

  Concepts Iceberg démontrés dans ce notebook :
  ✅ UPSERT ciblé via MERGE INTO (30 lignes sur 500 modifiées)
  ✅ Changelog inter-snapshots (lire uniquement les changements)
  ✅ Enrichissement par jointures multi-tables

  ➡  Prochain : 03_analytique_gold.ipynb


In [16]:
spark.stop()
print("✅ SparkSession fermée — Couche Silver complète !")

✅ SparkSession fermée — Couche Silver complète !
